## FIAP Connect — IA de Compatibilidade de Grupos
### Alexis Rondo (RM 560384) | Vinicius Rodrigues (RM 559611) | 2TDSPS

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Criação do dataframe com dados simulados do FIAP Connect
# Cada registro representa uma combinação aluno + grupo
np.random.seed(42)

registros = []

for _ in range(300):
    num_hab_aluno = np.random.randint(1, 8)
    num_faltantes = np.random.randint(1, 8)
    num_cobertas = np.random.randint(0, min(num_hab_aluno, num_faltantes) + 1)
    percentual = round((num_cobertas / num_faltantes) * 100, 1) if num_faltantes > 0 else 0
    mesmo_periodo = np.random.choice([0, 1], p=[0.3, 0.7])
    mesma_unidade = np.random.choice([0, 1], p=[0.4, 0.6])
    vagas = np.random.choice([1, 2])

    # Regras baseadas na lógica real do FIAP Connect
    if num_cobertas == num_faltantes and mesmo_periodo == 1:
        compat = 'ALTA'
    elif percentual >= 50 and mesmo_periodo == 1:
        compat = 'MEDIA'
    elif percentual >= 30 and mesma_unidade == 1:
        compat = 'MEDIA'
    else:
        compat = 'BAIXA'

    # Ruído pra simular incerteza real
    if np.random.random() < 0.08:
        opcoes = ['ALTA', 'MEDIA', 'BAIXA']
        opcoes.remove(compat)
        compat = np.random.choice(opcoes)

    registros.append([num_hab_aluno, num_faltantes, num_cobertas,
                      percentual, mesmo_periodo, mesma_unidade, vagas, compat])

colunas = ['num_habilidades_aluno', 'num_habilidades_faltantes_grupo',
           'num_cobertas', 'percentual_cobertura',
           'mesmo_periodo', 'mesma_unidade', 'vagas_disponiveis', 'compatibilidade']

df = pd.DataFrame(registros, columns=colunas)

In [ ]:
# Tamanho do dataframe
df.shape

In [ ]:
# Primeiros registros
df.head(10)

In [ ]:
# Informações dos atributos
df.info()

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Distribuição da variável alvo
df['compatibilidade'].value_counts()

In [ ]:
# Conversão de dados categóricos em numéricos
le = LabelEncoder()
df['compatibilidade_cod'] = le.fit_transform(df['compatibilidade'])

# Mapeamento
for i, classe in enumerate(le.classes_):
    print(f'{classe} -> {i}')

In [ ]:
# Visualização após transformação
df[['compatibilidade', 'compatibilidade_cod']].head()

In [ ]:
# Variável X (features)
X = df[['num_habilidades_aluno', 'num_habilidades_faltantes_grupo',
        'num_cobertas', 'percentual_cobertura',
        'mesmo_periodo', 'mesma_unidade', 'vagas_disponiveis']]
X.head()

In [ ]:
# Variável y (alvo)
y = df['compatibilidade_cod'].values
y

In [ ]:
# Separação 70% treino / 30% teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f'Treino: {X_train.shape[0]}')
print(f'Teste: {X_test.shape[0]}')

In [ ]:
# Modelo 1 - Random Forest
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

In [ ]:
# Predições do Random Forest
y_predict_rf = modelo_rf.predict(X_test)
y_predict_rf

In [ ]:
# Acurácia do Random Forest
acc_rf = accuracy_score(y_test, y_predict_rf)
print(f'Acurácia: {acc_rf*100:.2f}%')

In [ ]:
# Relatório do Random Forest
print(classification_report(y_test, y_predict_rf, target_names=le.classes_))

In [ ]:
# Modelo 2 - KNN
modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(X_train, y_train)

In [ ]:
# Predições do KNN
y_predict_knn = modelo_knn.predict(X_test)
y_predict_knn

In [ ]:
# Acurácia do KNN
acc_knn = accuracy_score(y_test, y_predict_knn)
print(f'Acurácia: {acc_knn*100:.2f}%')

In [ ]:
# Relatório do KNN
print(classification_report(y_test, y_predict_knn, target_names=le.classes_))

In [ ]:
# Comparação dos modelos
print(f'Random Forest: {acc_rf*100:.2f}%')
print(f'KNN:           {acc_knn*100:.2f}%')

melhor = 'Random Forest' if acc_rf >= acc_knn else 'KNN'
print(f'Melhor modelo: {melhor}')

In [ ]:
# Teste com dados reais do Oracle APEX
# Beatriz (RM333333) busca grupo - ela domina IOT e SQA

dados_apex = pd.DataFrame([
    # Beatriz vs Grupo 1 (FIAP Connect Team) - faltam IOT e SQA, ela cobre as 2
    {'num_habilidades_aluno': 2, 'num_habilidades_faltantes_grupo': 2,
     'num_cobertas': 2, 'percentual_cobertura': 100.0,
     'mesmo_periodo': 1, 'mesma_unidade': 1, 'vagas_disponiveis': 1},

    # Beatriz vs Grupo 2 (Oracle Innovators) - faltam 5, ela cobre 2
    {'num_habilidades_aluno': 2, 'num_habilidades_faltantes_grupo': 5,
     'num_cobertas': 2, 'percentual_cobertura': 40.0,
     'mesmo_periodo': 1, 'mesma_unidade': 1, 'vagas_disponiveis': 2}
])

melhor_modelo = modelo_rf if acc_rf >= acc_knn else modelo_knn
predicoes = melhor_modelo.predict(dados_apex)
labels = le.inverse_transform(predicoes)

print(f'Beatriz -> Grupo 1 (FIAP Connect Team):  {labels[0]}')
print(f'Beatriz -> Grupo 2 (Oracle Innovators):   {labels[1]}')

In [ ]:
# Salvando o modelo pra usar na Sprint 4 (API Flask)
with open('modelo_compatibilidade.pkl', 'wb') as f:
    pickle.dump(melhor_modelo, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('Modelo salvo: modelo_compatibilidade.pkl')
print('Encoder salvo: label_encoder.pkl')